---

## 1. Connection and Raw Table Verification

We connect to MySQL using credentials stored in `.env`. Using `readRenviron` rather than any external package because it is built into base R and handles `.env` files correctly. The password contains `$` and `@` — both are wrapped in double quotes in the `.env` file so the parser treats them as literals.

The first query retrieves basic row and column counts. This is the baseline check: we expect 56,635 rows based on the load step. If the count differs, something went wrong at load time and we stop here.

In [ ]:
library(DBI)
library(RMariaDB)
library(tidyverse)
library(scales)
library(janitor)

# Load credentials from .env
# readRenviron looks in the current working directory by default.
# If you get a credentials error, run getwd() and confirm it shows the project root.
readRenviron("../.env")

con <- dbConnect(
  RMariaDB::MariaDB(),
  host     = Sys.getenv("DB_HOST"),
  port     = as.integer(Sys.getenv("DB_PORT")),
  dbname   = Sys.getenv("DB_NAME"),
  user     = Sys.getenv("DB_USER"),
  password = Sys.getenv("DB_PASSWORD")
)

cat("Connection status:", dbIsValid(con), "\n")

In [ ]:
# Verify row count matches what was loaded
row_count <- dbGetQuery(con, "SELECT COUNT(*) AS total_rows FROM nashville_housing_raw")
print(row_count)

# Preview the first 5 rows to confirm structure
preview <- dbGetQuery(con, "SELECT * FROM nashville_housing_raw LIMIT 5")
glimpse(preview)

---

## 2. Column-Level Null Audit

Every column in the raw table is `VARCHAR`, so MySQL will not enforce any constraints. That means blank strings (`''`) and the string `'NULL'` can exist alongside genuine SQL `NULL` values. We need to count all three variants.

The query below generates a dynamic null audit across every column. For each column we count:
- True SQL NULLs
- Empty strings
- The literal string 'NULL' (common in CSV exports)

We then calculate the null rate as a percentage of total rows. Columns above 20% null are flagged for a decision: impute, exclude, or carry forward as-is with documentation.

In [ ]:
# Pull the full raw table into R for column-by-column analysis.
# We do this once and work in memory — avoids sending 29 separate queries to MySQL.
df_raw <- dbGetQuery(con, "SELECT * FROM nashville_housing_raw")

cat("Rows loaded:", nrow(df_raw), "\n")
cat("Columns loaded:", ncol(df_raw), "\n")

In [ ]:
# Null audit: count true NAs, empty strings, and the literal string 'NULL'
# across every column. This is important because CSVs don't always represent
# missing values consistently — we need to catch all three variants.

null_audit <- map_dfr(names(df_raw), function(col) {
  x <- df_raw[[col]]
  sql_null  <- sum(is.na(x))
  empty_str <- sum(!is.na(x) & x == "")
  str_null  <- sum(!is.na(x) & toupper(x) == "NULL")
  total     <- sql_null + empty_str + str_null
  tibble(
    column        = col,
    sql_null      = sql_null,
    empty_str     = empty_str,
    str_null      = str_null,
    total_missing = total,
    pct_missing   = round(total / nrow(df_raw) * 100, 2),
    flag          = case_when(
      total / nrow(df_raw) >= 0.20 ~ "HIGH",
      total / nrow(df_raw) >= 0.05 ~ "MODERATE",
      total > 0                    ~ "LOW",
      TRUE                         ~ "CLEAN"
    )
  )
}) %>%
  arrange(desc(total_missing))

print(null_audit)

In [ ]:
# Visualise null rates — columns with any missing values only
# Sorted descending so the worst offenders are immediately visible.

null_audit %>%
  filter(total_missing > 0) %>%
  ggplot(aes(x = reorder(column, pct_missing), y = pct_missing, fill = flag)) +
  geom_col() +
  geom_text(
    aes(label = paste0(pct_missing, "%")),
    hjust = -0.1, size = 3
  ) +
  coord_flip() +
  scale_fill_manual(values = c(
    "HIGH"     = "#c0392b",
    "MODERATE" = "#e67e22",
    "LOW"      = "#f1c40f"
  )) +
  scale_y_continuous(limits = c(0, 110), labels = label_percent(scale = 1)) +
  labs(
    title    = "Missing Value Rate by Column — Nashville Housing Raw",
    subtitle = "Includes SQL NULLs, empty strings, and literal 'NULL' strings",
    x        = NULL,
    y        = "% Missing",
    fill     = "Severity"
  ) +
  theme_minimal(base_size = 12) +
  theme(legend.position = "bottom")

---

## 3. Duplicate Detection

Duplicates are a financial risk: if we count a property sale twice, every aggregate metric is inflated. We check for duplicates at two levels:

- **Full row duplicates** — every column identical, including unique ID
- **Parcel ID duplicates** — same property identifier appearing more than once (legitimate re-sales vs data entry errors)

Note that a parcel appearing twice is not automatically an error — properties can be sold more than once in a four-year window. We will separate genuine re-sales (different sale dates) from true duplicates (identical sale date and price).

In [ ]:
# Full row duplicates
full_dupes <- df_raw %>%
  group_by(across(everything())) %>%
  filter(n() > 1) %>%
  ungroup()

cat("Full duplicate rows:", nrow(full_dupes), "\n")
cat("Unique duplicate sets:", nrow(full_dupes) / 2, "\n")

In [ ]:
# Parcel ID duplicates — split into genuine re-sales vs true duplicates
# A genuine re-sale: same ParcelID, different SaleDate
# A true duplicate: same ParcelID, same SaleDate, same SalePrice

# Full row duplicates
full_dupes <- df_raw %>%
  group_by(across(everything())) %>%
  filter(n() > 1) %>%
  ungroup()

cat("Full duplicate rows:", nrow(full_dupes), "\n")
cat("Unique duplicate sets:", nrow(full_dupes) / 2, "\n")

# Parcel ID duplicates — split into genuine re-sales vs true duplicates
parcel_dupes <- df_raw %>%
  group_by(parcel_id) %>%
  filter(n() > 1) %>%
  arrange(parcel_id, sale_date) %>%
  ungroup()

cat("Rows with a duplicate parcel_id:", nrow(parcel_dupes), "\n")

true_dupes <- parcel_dupes %>%
  group_by(parcel_id, sale_date, sale_price) %>%
  filter(n() > 1) %>%
  ungroup()

re_sales <- parcel_dupes %>%
  anti_join(true_dupes, by = c("parcel_id", "sale_date", "sale_price"))

cat("True duplicates (same parcel_id + sale_date + sale_price):", nrow(true_dupes), "\n")
cat("Genuine re-sales (same parcel_id, different sale_date):", nrow(re_sales), "\n")

---

## 4. Data Type Inspection and Casting Decisions

Every column arrived as `VARCHAR`. Before we can do any meaningful analysis, we need to know what each column actually contains and what type it should become. This section inspects the content of each column and documents the casting decision.

We pay particular attention to:
- `sale_price` — is it consistently numeric? Are there commas or currency symbols that would prevent casting?
- `sale_date` — what format is the date stored in? MySQL's `STR_TO_DATE` format string will depend on this.
- `year_built`, `Bedrooms`, `FullBath`, `HalfBath` — expect integer values but check for decimal entries or outliers
- `acreage`, `LandValue`, `BuildingValue`, `TotalValue` — expect numeric; check for formatting issues

In [ ]:
# Sample unique values from key columns to identify formatting issues before casting
columns_to_inspect <- c("sale_price", "sale_date", "year_built", "bedrooms",
                         "full_bath", "half_bath", "acreage", "land_value",
                         "building_value", "total_value")

for (col in columns_to_inspect) {
  cat("\n---", col, "---\n")
  sample_vals <- df_raw %>%
    filter(!is.na(.data[[col]]) & .data[[col]] != "") %>%
    pull(.data[[col]]) %>%
    unique() %>%
    head(10)
  print(sample_vals)
}

In [ ]:
# Check SalePrice specifically for non-numeric characters
# If any rows contain commas or symbols, they will fail a direct CAST in MySQL

price_issues <- df_raw %>%
  filter(!is.na(sale_price) & sale_price != "") %>%
  filter(grepl("[^0-9.]", sale_price)) %>%
  select(parcel_id, sale_price)

cat("sale_price rows with non-numeric characters:", nrow(price_issues), "\n")
if (nrow(price_issues) > 0) print(head(price_issues, 20))

In [ ]:
# Inspect SaleDate format
# We need to know the exact format before writing STR_TO_DATE() in the cleaning SQL

df_raw %>%
  filter(!is.na(sale_date) & sale_date != "") %>%
  pull(sale_date) %>%
  unique() %>%
  head(15)

---

## 5. Sale Price Distribution and Outlier Identification

Sale price is the dependent variable for every model in this project. Understanding its distribution before modelling is not optional. We are looking for:

- The overall shape — is it right-skewed (as property prices typically are)?
- Extreme outliers at the high end — genuine luxury sales, or data errors?
- Zero or near-zero values — inter-family transfers, tax sales, or recording errors?

The threshold for a statistical outlier here is defined using the IQR method: values below Q1 − 3×IQR or above Q3 + 3×IQR. We use 3×IQR rather than the standard 1.5×IQR because property markets have genuinely wide price variation and we do not want to flag legitimate high-value sales.

In [ ]:
# Cast SalePrice to numeric for analysis
# Rows that fail conversion (non-numeric content) will produce NA — we count those separately

prices <- df_raw %>%
  mutate(price_num = suppressWarnings(as.numeric(sale_price))) %>%
  filter(!is.na(price_num))

failed_cast <- df_raw %>%
  mutate(price_num = suppressWarnings(as.numeric(sale_price))) %>%
  filter(is.na(sale_price) | is.na(price_num))

cat("Rows with a valid numeric SalePrice:", nrow(prices), "\n")
cat("Rows that failed numeric conversion:", nrow(failed_cast), "\n")

In [ ]:
# Summary statistics for SalePrice
price_summary <- prices %>%
  summarise(
    n         = n(),
    min       = min(price_num),
    p25       = quantile(price_num, 0.25),
    median    = median(price_num),
    mean      = mean(price_num),
    p75       = quantile(price_num, 0.75),
    p95       = quantile(price_num, 0.95),
    max       = max(price_num),
    sd        = sd(price_num)
  )

print(price_summary)

In [ ]:
# IQR-based outlier detection (3x IQR for property data)
q1  <- quantile(prices$price_num, 0.25)
q3  <- quantile(prices$price_num, 0.75)
iqr <- q3 - q1

lower_bound <- q1 - 3 * iqr
upper_bound <- q3 + 3 * iqr

outliers <- prices %>%
  filter(price_num < lower_bound | price_num > upper_bound)

zero_or_near_zero <- prices %>%
  filter(price_num < 1000)

cat("Outliers by 3x IQR rule:", nrow(outliers), "\n")
cat("  Lower bound:", dollar(lower_bound), "\n")
cat("  Upper bound:", dollar(upper_bound), "\n")
cat("Zero or near-zero sales (< $1,000):", nrow(zero_or_near_zero), "\n")

In [ ]:
# Distribution plot — log scale on x-axis because property prices are right-skewed.
# Log scale makes the shape of the bulk of the distribution readable.
# The vertical lines mark the IQR outlier bounds.

prices %>%
  filter(price_num > 0) %>%
  ggplot(aes(x = price_num)) +
  geom_histogram(bins = 80, fill = "#2c3e50", alpha = 0.8) +
  geom_vline(xintercept = upper_bound, colour = "#e74c3c", linetype = "dashed") +
  geom_vline(xintercept = median(prices$price_num), colour = "#27ae60", linetype = "dashed") +
  scale_x_log10(labels = label_dollar()) +
  labs(
    title    = "Sale Price Distribution — Nashville Housing (2013–2016)",
    subtitle = "Log scale. Red: upper outlier bound (3x IQR). Green: median.",
    x        = "Sale Price (log scale)",
    y        = "Count"
  ) +
  theme_minimal(base_size = 12)

---

## 6. Date Range and Temporal Integrity

The dataset is described as 2013–2016. We verify that:
- All sale dates fall within that window
- No dates are in the future relative to the dataset period
- The distribution of sales across years is plausible (no single year has a suspicious spike or void)
- `YearBuilt` values are sensible — no buildings built before 1800 or after 2016

In [ ]:
# Parse SaleDate — adjust the format string if the inspection in Section 4 showed a different format
# Common formats: "%m/%d/%Y" for US dates like 01/15/2015, "%Y-%m-%d" for ISO format

dates <- df_raw %>%
  filter(!is.na(sale_date) & sale_date != "") %>%
  mutate(
    date_parsed = as.Date(sale_date, format = "%Y-%m-%d"),
    sale_year   = as.integer(format(date_parsed, "%Y"))
  )

failed_date_parse <- dates %>% filter(is.na(date_parsed))
cat("Rows that failed date parsing:", nrow(failed_date_parse), "\n")

cat("Date range:", format(min(dates$date_parsed, na.rm = TRUE)), "to",
    format(max(dates$date_parsed, na.rm = TRUE)), "\n")

In [ ]:
# Sales by year — look for unexpected gaps or spikes
dates %>%
  count(sale_year) %>%
  ggplot(aes(x = factor(sale_year), y = n)) +
  geom_col(fill = "#2c3e50") +
  geom_text(aes(label = comma(n)), vjust = -0.4, size = 3.5) +
  labs(
    title = "Sale Volume by Year",
    x     = "Year",
    y     = "Number of Sales"
  ) +
  theme_minimal(base_size = 12)

In [ ]:
# YearBuilt integrity check
year_built <- df_raw %>%
  filter(!is.na(year_built) & year_built != "") %>%
  mutate(yb = suppressWarnings(as.integer(year_built)))

cat("year_built range:", min(year_built$yb, na.rm = TRUE), "to",
    max(year_built$yb, na.rm = TRUE), "\n")

suspicious_years <- year_built %>%
  filter(yb < 1800 | yb > 2016)

cat("Suspicious year_built values (< 1800 or > 2016):", nrow(suspicious_years), "\n")

---

## 7. Categorical Field Standardisation Audit

Categorical columns often contain the same value written multiple ways — "Yes", "YES", "yes", "Y" — which will split what should be a single group into multiple groups in any analysis. We inspect the key categorical fields here and document what standardisation each one needs.

Fields to check: `SoldAsVacant`, `LandUse`, `OwnerCity`, `TaxDistrict`, `City`.

In [ ]:
categorical_cols <- c("sold_as_vacant", "land_use", "city", "tax_district", "property_city")

for (col in categorical_cols) {
  cat("\n---", col, "---\n")
  result <- df_raw %>%
    count(.data[[col]], sort = TRUE) %>%
    head(20)
  print(result)
}

---

## 8. Financial Impact Summary

This section attaches a dollar value to each data quality issue. The purpose is to demonstrate that data quality is a business problem, not a technical one. If we were to analyse the raw data without cleaning:

- True duplicates inflate total transaction volume and revenue aggregates
- Zero-value sales suppress median price calculations
- Outliers distort mean prices and model predictions
- Missing property characteristics make those rows useless for modelling

We quantify each of these impacts below.

In [ ]:
# Create prices_num here since Section 5 may not have run cleanly
prices_num <- df_raw %>%
  mutate(price_num = suppressWarnings(as.numeric(sale_price)))

# Impact of true duplicates
true_dupe_value <- prices_num %>%
  semi_join(true_dupes, by = c("parcel_id", "sale_date", "sale_price")) %>%
  summarise(total = sum(price_num, na.rm = TRUE)) %>%
  pull(total)

# Impact of zero/near-zero sales on median price
median_all     <- median(prices_num$price_num, na.rm = TRUE)
median_no_zero <- prices_num %>%
  filter(!is.na(price_num) & price_num >= 1000) %>%
  summarise(m = median(price_num, na.rm = TRUE)) %>%
  pull(m)

# Impact of outliers on mean price
mean_all         <- mean(prices_num$price_num, na.rm = TRUE)
mean_no_outliers <- prices_num %>%
  filter(!is.na(price_num) & price_num >= lower_bound & price_num <= upper_bound) %>%
  summarise(m = mean(price_num, na.rm = TRUE)) %>%
  pull(m)

cat("--- Financial Impact Summary ---\n\n")
cat("True duplicates:\n")
cat("  Rows:", nrow(true_dupes), "\n")
cat("  Inflated transaction value if counted:", dollar(true_dupe_value), "\n\n")

cat("Zero/near-zero sales (<$1,000):\n")
cat("  Count:", nrow(zero_or_near_zero), "\n")
cat("  Median price including zeros:", dollar(median_all), "\n")
cat("  Median price excluding zeros:", dollar(median_no_zero), "\n")
cat("  Distortion:", dollar(median_no_zero - median_all), "\n\n")

cat("Outliers (3x IQR):\n")
cat("  Count:", nrow(outliers), "\n")
cat("  Mean price including outliers:", dollar(mean_all), "\n")
cat("  Mean price excluding outliers:", dollar(mean_no_outliers), "\n")
cat("  Distortion:", dollar(mean_all - mean_no_outliers), "\n")

---

## 9. Produce Cleaned Table: `nashville_housing_clean`

The findings above inform the cleaning decisions applied here. This section does not re-derive any logic — it applies decisions already documented.

Cleaning steps applied:
- Remove true duplicates (keep one row per UniqueID)
- Cast `SalePrice`, `LandValue`, `BuildingValue`, `TotalValue`, `Acreage` to numeric
- Cast `YearBuilt`, `Bedrooms`, `FullBath`, `HalfBath` to integer
- Parse `SaleDate` to a proper date type
- Standardise `SoldAsVacant` to consistent values (Y / N)
- Trim and uppercase `LandUse`, `City`, `OwnerCity` to remove casing inconsistencies
- Flag outlier rows rather than removing them — a column `price_outlier_flag` is added so downstream analysis can exclude them where appropriate without losing rows permanently

This cleaning logic is written out in full in `sql/02_cleaning_queries.sql`. The SQL is the source of truth — this section writes the cleaned table to MySQL using that logic.

In [ ]:
df_clean <- df_raw %>%

  # Step 1: Remove true duplicates — keep first occurrence of each parcel+date+price combo
  distinct(parcel_id, sale_date, sale_price, .keep_all = TRUE) %>%

  # Step 2: Cast numeric fields
  mutate(
    sale_price     = suppressWarnings(as.numeric(sale_price)),
    land_value     = suppressWarnings(as.numeric(land_value)),
    building_value = suppressWarnings(as.numeric(building_value)),
    total_value    = suppressWarnings(as.numeric(total_value)),
    acreage        = suppressWarnings(as.numeric(acreage)),
    finished_area  = suppressWarnings(as.numeric(finished_area)),
    year_built     = suppressWarnings(as.integer(year_built)),
    bedrooms       = suppressWarnings(as.integer(bedrooms)),
    full_bath      = suppressWarnings(as.integer(full_bath)),
    half_bath      = suppressWarnings(as.integer(half_bath))
  ) %>%

  # Step 3: Parse sale_date — ISO format confirmed from data inspection
  mutate(
    sale_date = as.Date(sale_date, format = "%Y-%m-%d")
  ) %>%

  # Step 4: Standardise sold_as_vacant to Y / N
  mutate(
    sold_as_vacant = case_when(
      toupper(trimws(sold_as_vacant)) %in% c("YES", "Y") ~ "Y",
      toupper(trimws(sold_as_vacant)) %in% c("NO",  "N") ~ "N",
      TRUE ~ NA_character_
    )
  ) %>%

  # Step 5: Standardise text categoricals
  mutate(
  land_use                          = toupper(trimws(land_use)),
  property_city                     = toupper(trimws(property_city)),
  city                              = toupper(trimws(city)),
  grade                             = toupper(trimws(grade)),
  foundation_type                   = toupper(trimws(foundation_type)),
  exterior_wall                     = toupper(trimws(exterior_wall)),
  state                             = toupper(trimws(state)),
  multiple_parcels_involved_in_sale = toupper(trimws(multiple_parcels_involved_in_sale))
) %>%

  # Step 6: Add outlier flag
  mutate(
    price_outlier_flag = case_when(
      is.na(sale_price)                                          ~ "MISSING",
      sale_price < 1000                                          ~ "ZERO_OR_NEAR_ZERO",
      sale_price < lower_bound | sale_price > upper_bound        ~ "STATISTICAL_OUTLIER",
      TRUE                                                       ~ "CLEAN"
    )
  ) %>%

  # Step 7: Select and rename final columns
  select(
    parcel_id,
    land_use,
    property_address,
    suite_condo,
    property_city,
    sale_date,
    sale_price,
    price_outlier_flag,
    legal_reference,
    sold_as_vacant,
    multiple_parcels_involved_in_sale,
    owner_name,
    owner_address        = address,
    owner_city           = city,
    owner_state          = state,
    acreage,
    tax_district,
    neighborhood,
    land_value,
    building_value,
    total_value,
    finished_area,
    foundation_type,
    year_built,
    exterior_wall,
    grade,
    bedrooms,
    full_bath,
    half_bath
  )

cat("Cleaned rows:", nrow(df_clean), "\n")
cat("Removed (duplicates):", nrow(df_raw) - nrow(df_clean), "\n")
glimpse(df_clean)

In [ ]:
# Write cleaned table to MySQL
# overwrite = TRUE because if this cell is re-run, we want a fresh write, not an append

dbWriteTable(
  con,
  name      = "nashville_housing_clean",
  value     = df_clean,
  overwrite = TRUE,
  row.names = FALSE
)

# Verify the write
verify <- dbGetQuery(con, "SELECT COUNT(*) AS row_count FROM nashville_housing_clean")
cat("Rows in nashville_housing_clean:", verify$row_count, "\n")

In [ ]:
# Close the connection cleanly
dbDisconnect(con)
cat("Connection closed.\n")

## Audit Summary

| Issue | Count | Impact |
|---|---|---|
| Full row duplicates | 206 (103 unique sets) | Removed — one row per set retained |
| True duplicates (same parcel, date, price) | 334 rows | $67,568,326 inflated transaction value if counted |
| Genuine re-sales (same parcel, different date) | 14,850 rows | Retained — legitimate multi-sale parcels |
| Zero/near-zero sales (< $1,000) | 7 rows | $50 median distortion — flagged, not removed |
| Statistical outliers (3x IQR) | 1,904 rows | $84,115 mean distortion — flagged, not removed |
| Columns with HIGH null rate (> 20%) | 20 of 29 columns | Listed below |
| Rows in cleaned table | 56,468 | Down from 56,636 raw rows |

**Columns flagged HIGH (> 20% missing):**
`suite_condo` (89.2%), `half_bath` (57.4%), `bedrooms` (57.3%), `year_built` (57.3%), `grade` (57.3%), `foundation_type` (57.3%), `finished_area` (57.3%), `exterior_wall` (57.3%), `full_bath` (57.1%), `owner_name` (55.4%), `image` (55.3%), `total_value` (54.1%), `tax_district` (54.1%), `state` (54.1%), `neighborhood` (54.1%), `land_value` (54.1%), `city` (54.1%), `building_value` (54.1%), `address` (54.1%), `acreage` (54.1%)

**Key finding:** The ~54% null rate is consistent across all assessment-side fields (owner details, valuations, property characteristics). This is structural, not random — the dataset is likely a join of a transaction table and a property assessment table where the join failed for approximately half the records. Any analysis using these columns must be scoped to the 46% of records where they are present.

**Cleaning decisions:**
- Duplicates: removed using `distinct(parcel_id, sale_date, sale_price)` — keeps first occurrence of each unique transaction
- Outliers: flagged, not removed — `price_outlier_flag` column added with values `CLEAN`, `STATISTICAL_OUTLIER`, `ZERO_OR_NEAR_ZERO`, `MISSING`
- Zero/near-zero sales: flagged under `ZERO_OR_NEAR_ZERO` in `price_outlier_flag`
- Nulls in HIGH columns: carried forward as NA — no imputation applied at this stage
- Categorical standardisation: `toupper(trimws())` applied to `land_use`, `property_city`, `city`, `grade`, `foundation_type`, `exterior_wall`, `state`, `multiple_parcels_involved_in_sale`; `sold_as_vacant` standardised to Y / N
- `image` column excluded from cleaned table — Windows file paths with no analytical value